# Spam and Ham Project using Word2Vec and AvgWord2Vec
Importing and installing the required libraries.

In [1]:
! pip install gensim

import gensim
from gensim.models import Word2Vec, KeyedVectors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.6 MB/s eta 0:00:00


Loading the `Gensim` model.

In [2]:
import gensim.downloader as api

wv = api.load("word2vec-google-news-300")

vec_king = wv["king"]

[==================================================] 100.0% 1662.8/1662.8MB downloaded


Loading the dataset.

In [3]:
import pandas as pd
messages = pd.read_csv("/content/spam.csv")
messages.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1, inplace=True)
messages.rename({"v1":"label", "v2":"message"}, axis=1, inplace=True)
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


This time we will be using `Lemmatizer` instead of the `Stemmer`.

In [4]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [5]:
import re
import nltk
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [6]:
corpus = []
for i in range(len(messages)):
  review = re.sub("[^a-zA-Z]", " ", messages["message"][i])
  review = review.lower()
  review = review.split()
  review = [lemmatizer.lemmatize(word) for word in review]
  review = " ".join(review)
  corpus.append(review)

In [7]:
corpus

['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat',
 'ok lar joking wif u oni',
 'free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry question std txt rate t c s apply over s',
 'u dun say so early hor u c already then say',
 'nah i don t think he go to usf he life around here though',
 'freemsg hey there darling it s been week s now and no word back i d like some fun you up for it still tb ok xxx std chgs to send to rcv',
 'even my brother is not like to speak with me they treat me like aid patent',
 'a per your request melle melle oru minnaminunginte nurungu vettam ha been set a your callertune for all caller press to copy your friend callertune',
 'winner a a valued network customer you have been selected to receivea prize reward to claim call claim code kl valid hour only',
 'had your mobile month or more u r entitled to update to the latest colour mobile with camera for free call the mobile up

In [8]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

We will be using `simple_preprocess` from the **Gensim** which pre-processes the given data using the pre-trained model.

In [9]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
words = []
for sent in corpus:
  sent_token = sent_tokenize(sent)
  for sent in sent_token:
    words.append(simple_preprocess(sent))

Let's train this model.

In [11]:
import gensim

model = gensim.models.Word2Vec(sentences=words)

In [12]:
# Get the vocabulary size
model.corpus_count

5569

In [13]:
model.epochs

5

In [14]:
model.wv.similar_by_word("good")

[('hope', 0.9991095662117004),
 ('day', 0.9989493489265442),
 ('my', 0.9989069700241089),
 ('love', 0.9987552165985107),
 ('well', 0.998672604560852),
 ('great', 0.9986284375190735),
 ('did', 0.9986146092414856),
 ('dear', 0.9985958337783813),
 ('thing', 0.9985834956169128),
 ('much', 0.9985422492027283)]

In [15]:
model.wv["good"].shape

(100,)

In [16]:
words[0]

['go',
 'until',
 'jurong',
 'point',
 'crazy',
 'available',
 'only',
 'in',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'there',
 'got',
 'amore',
 'wat']

Let's perform `AvgWord2Vec`. As we know that the `Word2Vec` can be memory consuming, to save that we will use average approach.

In [17]:
import numpy as np

def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

Import `tqdm` for progress bar.

In [18]:
from tqdm import tqdm

X = []
for i in tqdm(range(len(words))):
  X.append(avg_word2vec(words[i]))

100%|██████████| 5569/5569 [00:00<00:00, 11438.36it/s]


In [19]:
X

[array([-0.20225409,  0.23711824,  0.16599862,  0.10830186,  0.12433972,
        -0.38227898,  0.24895176,  0.5421506 , -0.3263365 , -0.1206032 ,
        -0.15453084, -0.3716301 , -0.08131198,  0.06593578,  0.13240336,
        -0.13832374,  0.10952467, -0.3166413 ,  0.01471434, -0.51120776,
         0.1772449 ,  0.27758512,  0.07834803, -0.22112583, -0.06827475,
         0.07800665, -0.21564703, -0.17781426, -0.26705846,  0.04384015,
         0.20042707,  0.02864659,  0.16664378, -0.17506354, -0.06621345,
         0.3556177 , -0.03609397, -0.16897908, -0.14957613, -0.42834553,
         0.06724945, -0.22636193, -0.20813422, -0.04415166,  0.24245453,
        -0.05039973, -0.13497566, -0.12192474,  0.17772904,  0.16053315,
         0.13193159, -0.2407557 , -0.12299945, -0.01954261, -0.11019416,
         0.10825454,  0.09814119,  0.0188804 , -0.36699194,  0.15953523,
        -0.07062686,  0.07260083,  0.03681207, -0.12073113, -0.3345248 ,
         0.27857748,  0.16311914,  0.2893836 , -0.3

Here, we have completed the `AvgWord2Vec`.<br>
Now, we will consider the above as the *Independent Feature*.

In [20]:
X_new = np.array(X)
X_new.shape

(5569, 100)

In [21]:
X_new[0]

array([-0.20225409,  0.23711824,  0.16599862,  0.10830186,  0.12433972,
       -0.38227898,  0.24895176,  0.54215062, -0.3263365 , -0.1206032 ,
       -0.15453084, -0.3716301 , -0.08131198,  0.06593578,  0.13240336,
       -0.13832374,  0.10952467, -0.3166413 ,  0.01471434, -0.51120776,
        0.1772449 ,  0.27758512,  0.07834803, -0.22112583, -0.06827475,
        0.07800665, -0.21564703, -0.17781426, -0.26705846,  0.04384015,
        0.20042707,  0.02864659,  0.16664378, -0.17506354, -0.06621345,
        0.3556177 , -0.03609397, -0.16897908, -0.14957613, -0.42834553,
        0.06724945, -0.22636193, -0.20813422, -0.04415166,  0.24245453,
       -0.05039973, -0.13497566, -0.12192474,  0.17772904,  0.16053315,
        0.13193159, -0.24075571, -0.12299945, -0.01954261, -0.11019416,
        0.10825454,  0.09814119,  0.0188804 , -0.36699194,  0.15953523,
       -0.07062686,  0.07260083,  0.03681207, -0.12073113, -0.33452481,
        0.27857748,  0.16311914,  0.28938359, -0.34749964,  0.25

Let's go for the *Dependent Feature*.

In [22]:
y = pd.get_dummies(messages["label"])
y = y.iloc[:, 0:].values

In [23]:
messages.shape

(5572, 2)

If we take a look closely, we will observe that the number of records in `messages` is 3 more than that of `X`. Where are these 3 records?<br>
Basically, during the pre-processing only the sentences containing A-B and a-z. So the messages which doesn't contain these are deleted.

In [24]:
y = messages[list(map(lambda x : len(x) > 0, corpus))]
y = pd.get_dummies(y["label"])
y = y.iloc[:,0].values
y.shape

(5569,)

Let's go for finalizing the *Independent Features*. Let's create a DataFrame for storing the vector of each sentence in a single row.

In [25]:
X[0].reshape(1, -1).shape

(1, 100)

In [28]:
rows = []
for i in range(len(X)):
    rows.append(X[i].reshape(1, -1))

df = pd.concat([pd.DataFrame(r) for r in rows], ignore_index=True)

df.shape

(5569, 100)

In [29]:
df.head(5)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.202254,0.237118,0.165999,0.108302,0.124340,-0.382279,0.248952,0.542151,-0.326337,-0.120603,...,0.296052,0.202623,0.122542,0.155916,0.450262,0.262719,0.045271,-0.229499,0.152771,-0.068817
1,-0.184935,0.210561,0.146395,0.087376,0.112226,-0.336969,0.210854,0.476573,-0.286359,-0.102897,...,0.261503,0.170763,0.104459,0.136847,0.389278,0.235071,0.040170,-0.204965,0.128809,-0.063074
2,-0.202359,0.248714,0.180038,0.132853,0.132616,-0.418837,0.240277,0.546191,-0.349254,-0.109714,...,0.326161,0.202073,0.117074,0.156952,0.459091,0.255609,0.042966,-0.253055,0.165158,-0.059740
3,-0.280021,0.328379,0.223417,0.141347,0.165997,-0.520938,0.339724,0.738892,-0.450233,-0.170267,...,0.392128,0.263211,0.158515,0.220166,0.606154,0.360753,0.065353,-0.318563,0.199025,-0.088710
4,-0.243663,0.267728,0.193531,0.112846,0.147937,-0.436357,0.280767,0.619916,-0.372432,-0.153876,...,0.338085,0.217615,0.137152,0.185542,0.512647,0.309605,0.046829,-0.272440,0.169173,-0.076233


Independent Feature

In [30]:
X = df

Train Test Split

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [32]:
X_train.head(5)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
2675,-0.254785,0.319243,0.231828,0.149861,0.159072,-0.510431,0.338271,0.721824,-0.436229,-0.152367,...,0.399758,0.270471,0.167002,0.209424,0.599239,0.346209,0.063055,-0.292766,0.202293,-0.088375
3591,-0.250605,0.312401,0.231500,0.168074,0.165857,-0.538592,0.304988,0.697609,-0.437043,-0.142538,...,0.420152,0.257395,0.147930,0.193788,0.589244,0.329004,0.049625,-0.327439,0.211608,-0.076424
270,-0.249944,0.276567,0.200232,0.127255,0.156152,-0.460625,0.279362,0.639958,-0.393030,-0.143931,...,0.359753,0.226764,0.132445,0.184204,0.526540,0.307594,0.041622,-0.289425,0.186560,-0.064772
215,-0.190448,0.224063,0.159138,0.097913,0.122793,-0.362108,0.219680,0.496578,-0.310222,-0.105723,...,0.281000,0.184569,0.113256,0.144000,0.415063,0.242290,0.044717,-0.225897,0.159033,-0.061930
823,-0.206759,0.246619,0.179282,0.128386,0.132121,-0.414013,0.242969,0.553502,-0.354803,-0.111358,...,0.318234,0.210055,0.123960,0.161415,0.467433,0.264324,0.049141,-0.250339,0.165226,-0.064934


In [33]:
y

array([ True,  True, False, ...,  True,  True,  True])

Now, we are ready to apply any machine learning algorithm over this.

In [35]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

In [36]:
classifier.fit(X_train, y_train)

RandomForestClassifier()

Sometimes, a problem might arise that there are some `NaN` values. To solve that, we need to remove them from the dataset simply.<br>
Let's predict.

In [37]:
y_pred = classifier.predict(X_test)

In [38]:
from sklearn.metrics import accuracy_score, classification_report
print(accuracy_score(y_test, y_pred))

0.9766606822262118


In [39]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.92      0.91      0.91       148
        True       0.99      0.99      0.99       966

    accuracy                           0.98      1114
   macro avg       0.95      0.95      0.95      1114
weighted avg       0.98      0.98      0.98      1114

